## tl;dr

The published empirical survival model ranks the latest fully observed two-year cohort usefully, but calibration challengers do not beat the historical base-rate Brier score by the pre-declared 2% promotion margin. The current baseline should remain research-only while more independent REPD snapshots are collected.

## Context & Methods

This diagnostic companion reads the saved output from `analysis/forecast_audit.py`. The audit uses rolling-origin, project-purged holdouts and fits calibration only on cohorts earlier than the final test cohort.

### Key Assumptions

- REPD snapshots are the authoritative public project state.
- Outcomes are used only when the full two-, three- or five-year horizon is observable.
- A challenger is promoted only if it preserves positive ranking, has at least three rolling cohorts and improves Brier score versus the historical base rate by at least 2%.

## Data

In [ ]:
import json
from pathlib import Path
import pandas as pd

audit = json.loads(Path("analysis/forecast-audit.json").read_text())
pd.Series({key: audit[key] for key in ["latest_snapshot", "panel_rows", "projects", "snapshots"]})

## Results

In [ ]:
rows = []
for horizon, result in audit["horizons"].items():
    rows.append({
        "horizon_years": int(horizon),
        "test_cohort": result["test_cohort"],
        "test_rows": result["uncalibrated"]["rows"],
        "events": result["uncalibrated"]["events"],
        "raw_auc": result["uncalibrated"]["roc_auc"],
        "raw_brier": result["uncalibrated"]["brier_score"],
        "platt_brier": result["platt_calibrated"]["brier_score"],
        "base_brier": result["historical_base_rate"]["brier_score"],
        "brier_skill_vs_base": result["brier_skill_vs_base_rate"],
        "decision": result["promotion_decision"],
    })
results = pd.DataFrame(rows).sort_values("horizon_years")
results.style.format({"raw_auc": "{:.3f}", "raw_brier": "{:.4f}", "platt_brier": "{:.4f}", "base_brier": "{:.4f}", "brier_skill_vs_base": "{:.2%}"})

## Takeaways

- Keep the published empirical survival baseline and research-only label.
- Treat the two-year score primarily as a ranking signal; its latest holdout discrimination is materially stronger than the longer horizons.
- Do not train political or inflation coefficients from only 15 independent time snapshots. Keep these inputs in the transparent scenario layer.
- Re-run the audit after each new quarterly REPD snapshot and promote a challenger only when the declared threshold is met.